# 07 · The inspection records: who inspected, and how the data was produced

First of five notebooks (07–11) on the quality of the inspection data itself, following
notebook 06's finding that the inspector moves a score about as much as the property does.
This one settles whether inspector identity is complete and consistent enough to study, and
documents process artifacts that could pass for property differences.

**Sources of inspector identity** (reconciled in `inspectors.py`):

1. Salesforce "Assessed By: Name" on the observation rows of `multitenant_observations.dta`.
   A system user field.
2. Code Compliance's July 2026 backfill, `CPAL - MISSING INSPECTOR DATA 2021-2025.xlsx`, typed
   by hand.

Outputs use inspector codes (I01, I02, … by inspection count). The data come from a public
information request, so names are not confidential, but codes keep the analysis about
practice rather than individuals until the team decides how to share it.

## Findings

Run 2026-10-05. Outputs are cleared before commit; the numbers below are from that run.

**Inspector identity is complete and consistent enough for reliability work, once the two
sources are reconciled.**

| Source | Inspections with a name |
|---|---|
| analytic file `multitenant_inspections.dta` | 34% (4% in FY2020-21) |
| Salesforce name on any observation row | 62% |
| Code Compliance July 2026 backfill | 99.8% |
| reconciled (`inspectors.py`) | 100% of 6,197 |

- **Salesforce names are a system field.** 46 spellings, no variants.
- **The backfill is typed by hand.** 115 spellings. Matching on shared inspections finds five
  spellings of Salesforce names:
  - two with the first name reduced to an initial;
  - one with a suffix added;
  - one with first and last name swapped;
  - one with a different surname, likely a name change.

  Together they cover 532 backfill inspections. Case differences are normalized. Each mapping
  comes from the data, and a pair that co-occurs without sharing a name token is rejected.
- **The sources agree.** On the 3,826 inspections both name, they agree 88.5% of the time
  after normalizing and 97.4% after aliasing. The 98 remaining conflicts (2.6%) name
  different people. Most are in FY2022-23 (33) and FY2024-25 (46). Salesforce is used for
  those.
- **117 names in all.** 37 have 30 or more inspections and cover 96.8% of inspections. 68
  names have 1–4 inspections (100 in total), and only 14% of those carry a score. That fits
  Code's explanation of cases opened in error by staff from other divisions.
- **A note for the pipeline.** The inspection-level collapse keeps the first observation
  row's assessor name. Names that appear only on later rows are lost: 3,835 inspections have
  a Salesforce name on some row, but only 2,121 keep one. Step 4B's backfill covers the gap
  in the do-file, but the saved analytic `.dta` doesn't include the backfill.
- **Practice shifts by year.**

  | Fiscal year | Citations per inspection | Mean score | Bottom-10 rate |
  |---|---|---|---|
  | FY2020-21 | 3.6 | 96.9 | 4.1% |
  | FY2021-22 | 5.6 | 95.9 | 6.5% |
  | FY2022-23 | 7.4 | 90.8 | 15.5% |
  | FY2023-24 | 4.8 | 94.6 | 10.0% |
  | FY2024-25 | 7.8 | 93.9 | 12.0% |

  58–75% of inspections carry an assessed date. Code attributes the gap to how the report
  was filtered.
- **The inspector pool turned over.** Active inspectors (5+ inspections in a year) rose from
  13 to 22. 29 inspectors first became active after FY2020-21, and 20 stopped before
  FY2024-25. The 10 busiest inspectors did 57% of all inspections.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

import holdout as h
import inspectors as ins

pd.set_option("display.float_format", "{:.3f}".format)
DATA = h.DATA

## 1. Coverage by source

In [ ]:
analytic = pd.read_stata(DATA / "multitenant_inspections.dta", convert_categoricals=False)
analytic["sr_number"] = analytic.sr_number.astype(str).str.strip()
sf = ins.salesforce_names()
bf = ins.backfill_names()
who = ins.inspector_table()

cov = pd.DataFrame({
    "analytic file (first row only)": analytic.assign(n=~analytic.assessor_name.isin(["", "NA"])).groupby("fiscal_year").n.mean(),
    "Salesforce, any observation row": analytic.assign(n=analytic.sr_number.isin(sf.index)).groupby("fiscal_year").n.mean(),
    "Code backfill": analytic.assign(n=analytic.sr_number.isin(bf.index)).groupby("fiscal_year").n.mean(),
    "reconciled": analytic.assign(n=analytic.sr_number.isin(who.dropna(subset=["inspector_id"]).sr_number)).groupby("fiscal_year").n.mean(),
})
cov.loc["all years"] = [
    (~analytic.assessor_name.isin(["", "NA"])).mean(),
    analytic.sr_number.isin(sf.index).mean(),
    analytic.sr_number.isin(bf.index).mean(),
    analytic.sr_number.isin(who.dropna(subset=["inspector_id"]).sr_number).mean(),
]
print(f"{len(analytic):,} inspections in the analytic file")
print(cov.round(3).to_string())
print("\ninspections with 2+ different Salesforce names on their rows:", int((sf.sf_n_names >= 2).sum()))

## 2. Are names consistent?

In [ ]:
raw_obs = pd.read_parquet(DATA / ins.OBS_CACHE).assessor_name.astype("string").str.strip()
raw_obs = raw_obs[~raw_obs.isin(["", "NA"]) & raw_obs.notna()]
raw_bf = pd.read_excel(DATA / ins.BACKFILL, usecols=["Inspector"]).Inspector.astype("string").str.strip().dropna()
print("Salesforce: distinct raw spellings", raw_obs.nunique(), "-> after case/punctuation normalising", ins.norm(raw_obs).nunique())
print("Backfill:   distinct raw spellings", raw_bf.nunique(), "-> after normalising", ins.norm(raw_bf).nunique())

aliases = who.attrs["aliases"]
code = who.dropna(subset=["inspector_id"]).drop_duplicates("inspector_name").set_index("inspector_name").inspector_id


def kind(a: str, b: str) -> str:
    A, B = a.split(), b.split()
    if sorted(A) == sorted(B):
        return "name order swapped"
    if A[:len(B)] == B or B[:len(A)] == A:
        return "suffix added or dropped"
    if A[-1] == B[-1] and len(A[0]) == 1:
        return "first name reduced to an initial"
    if A[0] == B[0]:
        return "surname differs (likely a name change)"
    return "other"


rows = [{"maps to": code.get(t), "backfill inspections remapped": int((bf == s).sum()), "variant": kind(s, t)}
        for s, t in aliases.items()]
print(f"\naliases learned from the data: {len(aliases)}")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
both = sf.join(bf, how="inner")
agree_raw = (both.sf_name == both.bf_name).mean()
agree_alias = (both.sf_name == both.bf_name.replace(aliases)).mean()
print(f"inspections named by both sources: {len(both):,}")
print(f"  agree after normalising: {agree_raw:.1%}; after aliasing: {agree_alias:.1%}")
conf = who[who.conflict]
print(f"  conflicts (different people): {len(conf)} ({len(conf) / len(both):.1%} of overlap)")
print(conf.assign(fy=conf.sr_number.map(analytic.set_index('sr_number').fiscal_year)).fy.value_counts().sort_index().to_string())

counts = who.inspector_name.value_counts()
print(f"\ndistinct inspectors: {counts.size}; with 30+ inspections: {(counts >= 30).sum()} "
      f"(covering {counts[counts >= 30].sum() / counts.sum():.1%} of inspections); "
      f"with 1-4 inspections: {(counts < 5).sum()} ({counts[counts < 5].sum()} inspections)")
few = who[who.inspector_name.map(counts) < 5].sr_number
sc = analytic.set_index("sr_number").score
print(f"  inspections by 1-4-inspection names that carry a score: {sc.reindex(few).notna().mean():.1%} "
      f"(all inspections: {analytic.score.notna().mean():.1%})")

## 3. Process artifacts by fiscal year

In [ ]:
parts = []
with pd.read_stata(DATA / ins.OBS, iterator=True, chunksize=20_000,
                   columns=["sr_number", "work_type", "date_assessed", "subject"]) as r:
    parts.extend(r)
obs = pd.concat(parts, ignore_index=True)
obs["sr_number"] = obs.sr_number.astype(str).str.strip()
per_sr = obs.groupby("sr_number").agg(
    work_types=("work_type", "nunique"),
    assessed=("date_assessed", lambda s: s.notna().any()),
    citations=("subject", lambda s: (s.fillna("").str.strip() != "").sum()),
)
a = analytic.join(per_sr, on="sr_number")
a["inspector_id"] = a.sr_number.map(who.set_index("sr_number").inspector_id)
by_fy = a.groupby("fiscal_year").agg(
    inspections=("sr_number", "size"),
    scored=("score", lambda s: s.notna().mean()),
    has_assessed_date=("assessed", "mean"),
    one_work_type=("work_types", lambda s: (s == 1).mean()),
    mean_work_types=("work_types", "mean"),
    citations_per_inspection=("citations", "mean"),
    mean_score=("score", "mean"),
    bottom10=("bottom10", "mean"),
    inspectors=("inspector_id", "nunique"),
)
print(by_fy.round(3).to_string())

In [ ]:
# Turnover: inspectors active in each fiscal year (5+ inspections that year)
act = a.dropna(subset=["inspector_id"]).groupby(["inspector_id", "fiscal_year"]).size().unstack(fill_value=0) >= 5
act = act[act.any(axis=1)]  # inspectors never reaching 5 in a year are not counted
print("active inspectors per year:", act.sum().to_dict())
first = act.idxmax(axis=1)
last = act.iloc[:, ::-1].idxmax(axis=1)
print("first active year:", first.value_counts().sort_index().to_dict())
print("last active year: ", last.value_counts().sort_index().to_dict())
share_top = a.inspector_id.value_counts(normalize=True)
print(f"inspections by the 10 busiest inspectors: {share_top.head(10).sum():.1%}")